# Stockholm POI interactive map

Plots the **enriched analysis CSV** of Stockholm points of interest. Hover for key information; click for more details; toggle categories on the right.

The map is also saved as a standalone HTML file. **Place `stockholm_poi_enriched_analysis.csv` in the same directory as this notebook.**

In [ ]:
%pip install pandas folium

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\Kim\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


## 1. Load and check the data

In [ ]:
import pandas as pd
from pathlib import Path

csv_path = Path("stockholm_poi_enriched_analysis.csv")
df = pd.read_csv(csv_path, low_memory=False)
print(f"Loaded {len(df):,} POIs with {len(df.columns)} columns")
print(df["analysis_group"].value_counts().to_string())
display(df[["name", "primary_subcategory", "latitude", "longitude", "opening_hours", "all_pois_within_500m"]].head())

Loaded 4,234 POIs with 79 columns
analysis_group
food_drink               2397
culture_heritage          653
parks_recreation          526
nightlife                 321
visitor_attraction        249
culture_entertainment      88


,name,primary_subcategory,latitude,longitude,opening_hours,all_pois_within_500m
0,Stockholms stadshus,townhall,59.327493,18.054885,NaN,77
1,Stockholms slott,attraction,59.326941,18.071628,NaN,231
2,Riksdagshuset,building,59.327658,18.068031,NaN,206
3,Historiska museet,museum,59.334846,18.090150,"Tu, Th-Su 11:00-17:00; We 11:00-20:00",63
4,Arvfurstens palats,castle,59.329034,18.067564,NaN,188


## 2. Build the interactive map

Run this cell to generate the map with tooltips, popups and layer filters.

**Basemap problem?** Set `BASEMAP = None` to show the markers on a neutral background without loading external map tiles. You can also try `"OpenStreetMap"` if your network permits it. The background still requires internet when using Esri or OSM.

In [ ]:
import folium
import html
from folium.plugins import MarkerCluster

BASEMAP = "Esri WorldTopoMap"  # or "OpenStreetMap" or None
map_center = [59.3293, 18.0686]
m = folium.Map(location=map_center, zoom_start=12, tiles=None, control_scale=True, prefer_canvas=True)
if BASEMAP == "Esri WorldTopoMap":
    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Topo_Map/MapServer/tile/{z}/{y}/{x}",
        attr="Tiles © Esri and contributors", name="Esri WorldTopoMap", control=False
    ).add_to(m)
elif BASEMAP == "OpenStreetMap":
    folium.TileLayer("OpenStreetMap", control=False).add_to(m)

colors = {
    "food_drink": "#ed8739", "nightlife": "#b45dd3",
    "visitor_attraction": "#c53a47", "culture_heritage": "#3467b3",
    "culture_entertainment": "#16a19b", "parks_recreation": "#388b55",
    "accommodation": "#7a5b39",
}
labels = {
    "food_drink": "Restaurants & cafés", "nightlife": "Bars & nightlife",
    "visitor_attraction": "Tourist attractions", "culture_heritage": "Culture & heritage",
    "culture_entertainment": "Arts & entertainment", "parks_recreation": "Parks & recreation",
    "accommodation": "Accommodation",
}

clusters = {}
for group in sorted(df["analysis_group"].dropna().unique()):
    layer = folium.FeatureGroup(
        name=f'{labels.get(group, group)} ({(df["analysis_group"] == group).sum()})',
        show=True
    ).add_to(m)
    clusters[group] = MarkerCluster(
        options={"disableClusteringAtZoom": 16, "maxClusterRadius": 35}
    ).add_to(layer)

def val(row, key):
    value = row.get(key)
    return "" if pd.isna(value) else str(value).strip()

def field(row, label, key):
    value = val(row, key)
    if not value:
        return ""
    return f'<div style="margin-top:4px"><b>{html.escape(label)}:</b> {html.escape(value)}</div>'

plotted = 0
for _, row in df.iterrows():
    lat, lon = pd.to_numeric(row.get("latitude"), errors="coerce"), pd.to_numeric(row.get("longitude"), errors="coerce")
    group = val(row, "analysis_group")
    if pd.isna(lat) or pd.isna(lon) or group not in clusters:
        continue

    name = val(row, "name") or "Unnamed location"
    kind = val(row, "primary_subcategory").replace("_", " ")
    address = " ".join(filter(None, [val(row, "addr:street"), val(row, "addr:housenumber")]))
    heading = f'<div style="font-size:14px;font-weight:bold">{html.escape(name)}</div>'
    tooltip_html = (
        '<div style="font:12px/1.45 Arial;max-width:310px">' + heading
        + f'<div><b>Type:</b> {html.escape(kind)}</div>'
        + (f'<div><b>Address:</b> {html.escape(address)}</div>' if address else '')
        + field(row, "Opening hours", "opening_hours")
        + field(row, "Wheelchair access", "wheelchair")
        + field(row, "POIs within 500 m", "all_pois_within_500m")
        + field(row, "Attractions within 500 m", "visitor_attraction_within_500m")
        + '</div>'
    )
    extra_fields = [
        ("OSM categories", "all_osm_categories"), ("Cuisine", "cuisine"),
        ("Operator", "operator"), ("Website", "best_website"),
        ("Phone", "best_phone"), ("Outdoor seating", "outdoor_seating"),
        ("Vegan", "diet:vegan"), ("Vegetarian", "diet:vegetarian"),
        ("Fee", "fee"), ("Description", "description"),
        ("Nearby restaurants (500 m)", "food_drink_within_500m"),
        ("Nearby nightlife (500 m)", "nightlife_within_500m"),
        ("Nearby heritage (500 m)", "culture_heritage_within_500m"),
        ("Total POIs (1 km)", "all_pois_within_1000m"),
    ]
    details = "".join(field(row, label, key) for label, key in extra_fields)
    osm = val(row, "osm_url")
    if osm.startswith("https://www.openstreetmap.org/"):
        details += f'<div style="margin-top:8px"><a href="{html.escape(osm, quote=True)}" target="_blank">OpenStreetMap ↗</a></div>'
    popup_html = f'<div style="font:12px/1.45 Arial;min-width:250px">{heading}{details}</div>'

    folium.CircleMarker(
        [lat, lon], radius=5, color="white", weight=1,
        fill=True, fill_color=colors.get(group, "#677380"), fill_opacity=.9,
        tooltip=folium.Tooltip(tooltip_html, sticky=True),
        popup=folium.Popup(popup_html, max_width=400),
    ).add_to(clusters[group])
    plotted += 1

folium.LayerControl(collapsed=False).add_to(m)
print(f"Plotted {plotted:,} locations")

Plotted 4,234 locations


## 3. Save and view the map

Open the generated HTML file in your browser for the best experience. In the notebook, `display(m)` can also render it if the notebook is trusted.

In [ ]:
from IPython.display import display, HTML
output_path = Path("stockholm_poi_interactive.html")
m.save(str(output_path))
print("Saved:", output_path.resolve())
# Uncomment to render inside Jupyter (trust the notebook if prompted):
# display(m)


Saved: C:\Users\Kim\software engineering\year 5\LP1\4DT911\4dt911-vis-project-research\notebooks\data_preprocessing\stockholm_poi_interactive.html
